In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS medical_pipeline.silver;

In [0]:
from pyspark.sql.functions import col
import re
from pyspark.sql.functions import to_timestamp

df_bronze = spark.table("`medical_pipeline`.bronze.encounters")

df_silver = df_bronze.select("id","patient","payer","code","description","encounter_class" ,"start" , "stop" , "base_encounter_cost","total_claim_cost", "payer_coverage" , "reasoncode","reasondescription")

df_silver = df_silver.withColumn("start", to_timestamp("start")) \
               .withColumn("stop", to_timestamp("stop"))

df_silver = df_silver.fillna({
    "encounter_class": "unknown",
    
    "reasondescription" : "unknown"
})

df_silver = df_silver.dropDuplicates(["id"])

df_silver = df_silver.filter(
    col("id").isNotNull() &
    col("patient").isNotNull() &
    col("start").isNotNull()
)

In [0]:

df_silver.write \
  .mode("overwrite") \
  .format("delta") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medical_pipeline.silver.encounters_silver")


In [0]:
%sql
select * from medical_pipeline.silver.encounters_silver;